In [ ]:
# Sistema legado: contraseñas guardadas SIN protección (inseguro)
legado = [
    {"usuario": "osmith",    "correo": "oliver.smith@tienda.com",    "clave": "Oliver2024"},
    {"usuario": "msuarez",   "correo": "mauricio.suarez@tienda.com", "clave": "Camba1234"},
    {"usuario": "ebrown",    "correo": "emily.brown@tienda",         "clave": "Emily2025"},   # correo inválido
    {"usuario": "jroca",     "correo": "jimena.roca@tienda.com",     "clave": "123"},         # clave corta
    {"usuario": "osmith",    "correo": "otro.smith@tienda.com",      "clave": "Repetido99"},  # usuario repetido
    {"usuario": "hwilliams", "correo": "harry.williams@tienda.com",  "clave": "Harry7890"},
    {"usuario": "",          "correo": "sin.usuario@tienda.com",     "clave": "Vacio1234"},   # usuario vacío
]

print(f"Registros en el sistema legado: {len(legado)}")

In [ ]:
import hashlib
import hmac
import os

ITERACIONES = 100_000

def generar_hash(clave, sal=None):
    """Convierte la contraseña en un hash con sal. La contraseña original nunca se guarda."""
    if sal is None:
        sal = os.urandom(16)  # sal aleatoria y distinta para cada usuario
    hash_clave = hashlib.pbkdf2_hmac("sha256", clave.encode("utf-8"), sal, ITERACIONES)
    return sal.hex(), hash_clave.hex()

def verificar_clave(clave, sal_hex, hash_guardado):
    """Calcula el hash de la clave ingresada y lo compara con el guardado."""
    _, hash_nuevo = generar_hash(clave, bytes.fromhex(sal_hex))
    return hmac.compare_digest(hash_nuevo, hash_guardado)

In [ ]:
import re

PATRON_CORREO = r"^[\w.+-]+@[\w-]+(\.[\w-]+)*\.[a-zA-Z]{2,}$"

def validar_registro(registro, usuarios_vistos):
    """Devuelve una lista de errores. Si está vacía, el registro es válido."""
    errores = []
    usuario = registro["usuario"].strip()
    correo = registro["correo"].strip()
    clave = registro["clave"]

    if not usuario:
        errores.append("usuario vacío")
    elif usuario in usuarios_vistos:
        errores.append("usuario repetido")

    if not re.match(PATRON_CORREO, correo):
        errores.append("correo con formato inválido")

    if len(clave) < 8:
        errores.append("contraseña con menos de 8 caracteres")
    elif not re.search(r"\d", clave):
        errores.append("contraseña sin números")

    return errores

In [ ]:
import sqlite3

conexion = sqlite3.connect("sistema_seguro.db")
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS usuarios")
cursor.execute("""
CREATE TABLE usuarios (
    usuario   TEXT    NOT NULL,
    correo    TEXT    NOT NULL,
    sal       TEXT    NOT NULL,
    hash      TEXT    NOT NULL,
    intentos  INTEGER NOT NULL DEFAULT 0,
    bloqueado INTEGER NOT NULL DEFAULT 0,
    CONSTRAINT pk_usuarios PRIMARY KEY (usuario),
    CONSTRAINT ck_usuarios_intentos CHECK (intentos >= 0),
    CONSTRAINT ck_usuarios_bloqueado CHECK (bloqueado IN (0, 1))
)
""")

usuarios_vistos = set()
migrados = 0
rechazados = 0

print("=== REPORTE DE MIGRACIÓN ===\n")

for registro in legado:
    nombre = registro["usuario"].strip() or "(vacío)"
    errores = validar_registro(registro, usuarios_vistos)

    if errores:
        rechazados += 1
        print(f"❌ {nombre}: {', '.join(errores)}")
        continue

    sal, hash_clave = generar_hash(registro["clave"])

    # Consulta parametrizada (?): evita inyección SQL
    cursor.execute(
        "INSERT INTO usuarios (usuario, correo, sal, hash) VALUES (?, ?, ?, ?)",
        (registro["usuario"].strip(), registro["correo"].strip(), sal, hash_clave)
    )
    usuarios_vistos.add(registro["usuario"].strip())
    migrados += 1
    print(f"✅ {nombre}: migrado | contraseña: ********")

conexion.commit()
print(f"\nMigrados: {migrados} | Rechazados: {rechazados}")

In [ ]:
print(f"{'Usuario':<10} | {'Correo':<27} | {'Hash (inicio)':<23} | Intentos | Bloqueado")
print("-" * 85)

for usuario, correo, hash_clave, intentos, bloqueado in cursor.execute(
    "SELECT usuario, correo, hash, intentos, bloqueado FROM usuarios"
):
    print(f"{usuario:<10} | {correo:<27} | {hash_clave[:20]}... | {intentos:^8} | {bloqueado:^9}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

MAX_INTENTOS = 3

titulo = widgets.HTML("<h2 style='color: #2B548A;'>🔐 INICIO DE SESIÓN</h2>")
txt_usuario = widgets.Text(description="Usuario:")
txt_clave = widgets.Password(description="Contraseña:")
btn_ingresar = widgets.Button(description="Ingresar", button_style="success", icon="sign-in")
salida = widgets.Output()

def iniciar_sesion(b):
    with salida:
        clear_output()
        usuario = txt_usuario.value.strip()
        clave = txt_clave.value
        txt_clave.value = ""  # la contraseña no queda escrita en pantalla

        if not usuario or not clave:
            print("⚠️ Completa usuario y contraseña.")
            return

        cursor.execute(
            "SELECT sal, hash, intentos, bloqueado FROM usuarios WHERE usuario = ?",
            (usuario,)
        )
        fila = cursor.fetchone()

        # Mensaje genérico: no revela si el usuario existe
        if fila is None:
            print("❌ Usuario o contraseña incorrectos.")
            return

        sal, hash_guardado, intentos, bloqueado = fila

        if bloqueado:
            print("🔒 Cuenta bloqueada por demasiados intentos fallidos.")
            return

        if verificar_clave(clave, sal, hash_guardado):
            cursor.execute("UPDATE usuarios SET intentos = 0 WHERE usuario = ?", (usuario,))
            conexion.commit()
            print(f"✅ Bienvenido, {usuario}.")
        else:
            intentos += 1
            bloquear = 1 if intentos >= MAX_INTENTOS else 0
            cursor.execute(
                "UPDATE usuarios SET intentos = ?, bloqueado = ? WHERE usuario = ?",
                (intentos, bloquear, usuario)
            )
            conexion.commit()
            if bloquear:
                print("🔒 Cuenta bloqueada por demasiados intentos fallidos.")
            else:
                print("❌ Usuario o contraseña incorrectos.")

btn_ingresar.on_click(iniciar_sesion)
display(titulo, txt_usuario, txt_clave, btn_ingresar, salida)